# Новый домен: сравнить четыре компактных детектора

**На занятии:** проверить разметку, настроить обучение, выполнить по две пробные
эпохи для каждой из четырёх моделей и разобрать результаты. **Самостоятельно:**
зафиксировать настройки и запустить каждую модель на 20 эпох, затем сравнить результаты.

Домен — микроскопические изображения крови. Классы: эритроциты `RBC`, лейкоциты
`WBC`, тромбоциты `Platelets`. Это другие изображения и классы, чем у COCO-весов.

Каждый студент обучает и сравнивает **все четыре модели**:

| Модель | Семейство | Назначение в сравнении |
|---|---|---|
| SSDLite320 | Якорный SSD / MobileNetV3 | Компактный детектор другого семейства |
| YOLOv5n | Якорный YOLO | Исходная архитектура YOLOv5 |
| YOLO11n | Безъякорный YOLO | Компактная модель следующего поколения |
| YOLO26n | Безъякорный YOLO | Более новое поколение с двумя обучаемыми головами |

У всех вход 320 и batch по умолчанию 2. SSDLite использует MobileNetV3 и default
boxes. YOLOv5n использует исходную якорную голову; это **не YOLOv5u**.
YOLO11 и YOLO26 — безъякорные. Число параметров считаем после замены классов.
Все четыре модели начинают с предобученных весов. Они обучаются **последовательно**:
одновременно держать четыре модели на видеокарте не требуется.

Для YOLO26 используем ветку one-to-many с обычным NMS. Его отдельная
возможность инференса без NMS в этом сравнении не включена.

Нужен GPU для обучения. На CPU можно подготовить данные и проверить короткий
запуск вручную, но 20 эпох лучше выполнить в Colab с GPU или локально на NVIDIA.
Все переключатели обучения сначала выключены: Run All не запускает длинный опыт.
Перед обучением реализуйте **три блока TODO**: преобразование рамок, настройку
обучаемых параметров и анализ ошибок. Проверки остановят выполнение на незавершённом
блоке. Циклы обучения четырёх моделей уже подготовлены.

Функции TODO должны зависеть от своих аргументов и библиотек `np` / `torch` /
`pd`. Вспомогательные вычисления размещайте внутри соответствующей функции:
так ваш код можно использовать и в отдельном процессе обучения YOLOv5.

## 1. Подготовка и план опыта

Откройте этот ноутбук отдельно в Colab или Jupyter и выполняйте сверху вниз.
В Colab выберите GPU до начала работы. Сначала заполните гипотезу о различиях
моделей и прочитайте четыре настройки. Пока оставьте `RUN_PILOT=False` и `RUN_LONG=False`.

**Короткие вопросы.** Почему нельзя сравнивать номер класса 0 у исходного COCO
детектора с номером 0 нового датасета? Что должно измениться в классификаторе?
Что экономит заморозка backbone и чему она может помешать в новом домене?

In [ ]:
# Выполните до остальных ячеек. В Colab готовый PyTorch обычно уже установлен.
import importlib.metadata as meta
import subprocess
import sys

packages = ['ultralytics==8.4.102', 'numpy>=1.26,<3',
            'matplotlib>=3.8,<4', 'pandas>=2.1,<4', 'Pillow>=10,<13',
            'ipywidgets>=8.1,<9']
try:
    ready = meta.version('ultralytics') == '8.4.102'
    for package in ['numpy', 'matplotlib', 'pandas', 'Pillow', 'torch', 'torchvision', 'ipywidgets']:
        meta.version(package)
except meta.PackageNotFoundError:
    ready = False
if not ready:
    # Сохраняем уже установленную пару PyTorch/torchvision, включая CUDA-сборку.
    # Если есть только torch, resolver подберёт совместимую torchvision к этой версии.
    for package in ['torch', 'torchvision']:
        try:
            packages.append(package + '==' + meta.version(package))
        except meta.PackageNotFoundError:
            pass
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
try:
    from google.colab import output as colab_output
except ImportError:
    pass
else:
    colab_output.enable_custom_widget_manager()
print('Библиотеки готовы. Python:', sys.version.split()[0])

# Дополнительные библиотеки исходного YOLOv5; готовую пару torch/torchvision сохраняем.
extra_packages = {'scipy': 'scipy>=1.11,<2', 'seaborn': 'seaborn>=0.13,<0.14',
                  'tqdm': 'tqdm>=4.66,<5', 'GitPython': 'GitPython>=3.1.30,<4'}
missing = []
for package, requirement in extra_packages.items():
    try:
        meta.version(package)
    except meta.PackageNotFoundError:
        missing.append(requirement)
if missing:
    # Не обновляем уже установленные основные библиотеки при установке дополнений.
    preserved = [package + '==' + meta.version(package) for package in
                 ['torch', 'torchvision', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'ultralytics']]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing, *preserved])
print('Дополнения для обучения готовы.')

In [ ]:
from pathlib import Path
import os, json, time, hashlib, shutil, urllib.request, zipfile, random, gc
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display
import torch, torchvision, ultralytics
from ultralytics import YOLO

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(min(4, os.cpu_count() or 1))
ROOT = Path(os.environ.get('DETECTION_LAB_ROOT', 'detection_lab_data')).resolve() / 'domain_transfer'
ROOT.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ['RBC', 'WBC', 'Platelets']
LOW_CONF, MATCH_IOU, NMS_IOU, MAX_DET = 0.01, 0.5, 0.5, 100
print('Устройство:', DEVICE, '| torch:', torch.__version__, '| torchvision:', torchvision.__version__)
if DEVICE.startswith('cuda'):
    print(torch.cuda.get_device_name(0))

In [ ]:
# 1. В работе сравниваются все четыре модели; они обучаются последовательно.
STUDENT_ID = 'my_experiment'  # короткий псевдоним для своих результатов
HYPOTHESIS = ''  # Какую разницу ожидаете и по какой метрике проверите?

# 2. Перед первым запуском объясните назначение этих четырёх настроек.
TRAIN_CONFIG = dict(batch_size=2, lr=0.001, freeze_backbone=True, augment_hflip=True)
SEED = 42
PILOT_EPOCHS = 2
LONG_EPOCHS = 20
RUN_PILOT = False
RUN_LONG = False
RUN_TEST = False
ALLOW_CPU_TRAINING = False
PILOT_NOTE = ''  # Что показал пилот? Что исправили/почему оставили настройки?
FINAL_CONFIG_NOTE = ''  # Почему эти настройки готовы к 20 эпохам?

SELECTED_MODELS = ['ssdlite320', 'yolov5n', 'yolo11n', 'yolo26n']
MODEL_INFO = {
    'ssdlite320': dict(architecture='SSDLite / MobileNetV3', anchors=True, inference='NMS'),
    'yolov5n': dict(architecture='YOLOv5 original', anchors=True, inference='NMS'),
    'yolo11n': dict(architecture='YOLO11 nano', anchors=False, inference='NMS'),
    'yolo26n': dict(architecture='YOLO26 nano', anchors=False, inference='one-to-many + NMS'),
}
# В Colab при необходимости задайте путь внутри подключённого Google Drive.
ARTIFACTS = (Path('artifacts/domain_transfer') / hashlib.sha256(STUDENT_ID.encode()).hexdigest()[:12]).resolve()
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Модели:', SELECTED_MODELS, '| результаты:', ARTIFACTS)

## 2. Данные: сначала увидеть рамки

В BCCD 364 изображения. Используем готовое разбиение, но исключаем один почти
повторный кадр из val, сцена которого есть в train: остаются **205 train / 86 val /
72 test**. Две точечные аннотации исключаются по заранее заданному правилу.
Журнал очистки сохраняется вместе с результатами; остальные рамки не правим по
результатам модели. Это разбиение по изображениям: независимость пациентов или
стёкол им не гарантируется.

Тромбоцитов заметно меньше, чем эритроцитов. Поэтому сравниваем AP и recall
**отдельно по классам**, а затем невзвешенное среднее AP. Хороший суммарный счёт
по большинству объектов может скрывать плохую работу с редким классом.

In [ ]:
ARCHIVE_SHA = 'e2ed144a0a2c6dbe351f3ddb09fb8ff2c9ad4f947bdf42b6a52dd4f38a996ba9'
COMMIT = 'd272fb14cdff6e473fafeeeba32aba5f560e9e43'

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def download_checked(url, target, expected):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and sha256_file(target) == expected:
        return target
    temporary = target.with_suffix(target.suffix + '.part')
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'DetectionPractical/1.0'})
        with urllib.request.urlopen(request, timeout=120) as src, temporary.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        if sha256_file(temporary) != expected:
            raise ValueError('Контрольная сумма не совпала: ' + target.name)
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return target

archive = download_checked(
    f'https://codeload.github.com/Shenggan/BCCD_Dataset/zip/{COMMIT}',
    ROOT / 'bccd.zip', ARCHIVE_SHA)
DATA = ROOT / f'BCCD_Dataset-{COMMIT}' / 'BCCD'
if not (DATA / '.complete').exists():
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT):
                raise ValueError('Некорректный путь в архиве')
        z.extractall(ROOT)
    (DATA / '.complete').touch()

cleaning_log = []
def read_domain_item(stem):
    image_path = DATA / 'JPEGImages' / (stem + '.jpg')
    xml = ET.parse(DATA / 'Annotations' / (stem + '.xml')).getroot()
    with Image.open(image_path) as im:
        width, height = im.size
    boxes, labels = [], []
    for obj in xml.findall('object'):
        name = obj.findtext('name')
        assert name in CLASS_NAMES, name
        b = obj.find('bndbox')
        x1, y1, x2, y2 = [float(b.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
        # В VOC координаты включают обе границы и начинаются с 1.
        # До перевода исключаем две точечные аннотации по фиксированному правилу.
        if x2 <= x1 or y2 <= y1:
            cleaning_log.append(dict(image=stem, reason='point_annotation', class_name=name))
            continue
        box = np.array([x1-1, y1-1, x2, y2], dtype=float)
        box[[0, 2]] = np.clip(box[[0, 2]], 0, width)
        box[[1, 3]] = np.clip(box[[1, 3]], 0, height)
        assert box[2] > box[0] and box[3] > box[1]
        boxes.append(box)
        labels.append(CLASS_NAMES.index(name))
    return dict(name=image_path.name, path=image_path, width=width, height=height,
                boxes=np.asarray(boxes, dtype=float).reshape(-1, 4), labels=np.asarray(labels, dtype=int))

all_items = {p.stem: read_domain_item(p.stem) for p in sorted((DATA / 'JPEGImages').glob('*.jpg'))}
assert len(all_items) == 364 and len(cleaning_log) == 2
splits = {}
for split in ['train', 'val', 'test']:
    stems = (DATA / 'ImageSets' / 'Main' / (split + '.txt')).read_text().split()
    # Этот почти повторный кадр совпадает со сценой 00038 из train.
    stems = [stem for stem in stems if stem != 'BloodImage_00037']
    splits[split] = sorted(stems)
assert not (set(splits['train']) & set(splits['val']))
assert not (set(splits['train'] + splits['val']) & set(splits['test']))
train_items, val_items, test_items = [[all_items[s] for s in splits[k]] for k in ['train', 'val', 'test']]
assert [len(train_items), len(val_items), len(test_items)] == [205, 86, 72]
rows = []
for split, subset in [('train', train_items), ('val', val_items), ('test', test_items)]:
    labels = np.concatenate([x['labels'] for x in subset])
    rows.append(dict(split=split, images=len(subset), **{name: int(np.sum(labels == i)) for i, name in enumerate(CLASS_NAMES)}))
display(pd.DataFrame(rows))
display(pd.DataFrame(cleaning_log))
(ARTIFACTS / 'split.json').write_text(json.dumps(splits, indent=2))
(ARTIFACTS / 'cleaning.json').write_text(json.dumps(cleaning_log + [dict(image='BloodImage_00037', reason='near_duplicate_of_train_00038')], indent=2))

### Реализуйте 1/3: координаты и отражение

Заполните две функции в следующей ячейке. Их использует подготовка данных для
реального обучения, поэтому сначала добейтесь прохождения проверок.

- `box_to_yolo(box, width, height)` принимает `[x1, y1, x2, y2]` в пикселях и
  возвращает NumPy-массив `[cx/W, cy/H, w/W, h/H]`. Координаты полуоткрытые:
  ширина рамки равна `x2 - x1`, без добавления единицы. Не обрезайте рамку молча.
- `hflip_sample(image, boxes)` принимает изображение NumPy `H×W×C` и рамки
  `N×4`, отражает оба по горизонтали и возвращает **новые** массивы. Для пустого
  набора рамок сохраните форму `(0, 4)`. Входные массивы менять нельзя.

**Сначала предскажите:** как изменятся центр и ширина рамки после отражения?
Почему при переходе к YOLO нельзя делить все четыре числа на ширину изображения?
После проверок выберите реальный train-кадр и наложите исходные и отражённые
рамки на изображения: численный тест не заменяет просмотр разметки.

In [ ]:
def box_to_yolo(box, width, height):
    """Пиксельная xyxy-рамка → нормированные cx, cy, width, height."""
    # TODO: вычислите центр и размеры, нормируйте каждую ось своим размером.
    raise NotImplementedError('Реализуйте box_to_yolo в задании 1/3.')


def hflip_sample(image, boxes):
    """Отразить HWC-изображение и N×4 xyxy-рамки, не меняя входы."""
    # TODO: отразите пиксели, затем пересчитайте левую и правую границы рамок.
    raise NotImplementedError('Реализуйте hflip_sample в задании 1/3.')

In [ ]:
# Эти примеры проверяют договорённость о координатах, а не качество обучения.
_test_box = np.array([20., 10., 100., 50.])
_box_copy = _test_box.copy()
_converted = box_to_yolo(_test_box, width=200, height=80)
assert isinstance(_converted, np.ndarray) and _converted.shape == (4,), 'Верните массив формы (4,).'
np.testing.assert_allclose(_converted, [0.3, 0.375, 0.4, 0.5], atol=1e-7,
                           err_msg='Перепроверьте нормирование осей и полуоткрытые координаты.')
np.testing.assert_array_equal(_test_box, _box_copy, err_msg='box_to_yolo изменил входную рамку.')
np.testing.assert_allclose(box_to_yolo([0, 0, 200, 80], 200, 80), [.5, .5, 1., 1.])

_image = np.arange(4 * 7 * 3, dtype=np.uint8).reshape(4, 7, 3)
_boxes = np.array([[1., 0., 4., 2.], [0., 1., 7., 4.]])
_image_copy, _boxes_copy = _image.copy(), _boxes.copy()
_flipped, _flipped_boxes = hflip_sample(_image, _boxes)
assert _flipped.shape == _image.shape and _flipped.dtype == _image.dtype, 'Сохраните форму и dtype изображения.'
np.testing.assert_array_equal(_flipped[:, 0], _image[:, -1], err_msg='Отражать нужно горизонтальную ось.')
np.testing.assert_array_equal(_flipped_boxes, [[3., 0., 6., 2.], [0., 1., 7., 4.]])
np.testing.assert_array_equal(_image, _image_copy, err_msg='Изменён входной массив изображения.')
np.testing.assert_array_equal(_boxes, _boxes_copy, err_msg='Изменён входной массив рамок.')
assert not np.shares_memory(_image, _flipped), 'Верните независимую копию изображения.'
assert not np.shares_memory(_boxes, _flipped_boxes), 'Верните независимую копию рамок.'
_twice_image, _twice_boxes = hflip_sample(_flipped, _flipped_boxes)
np.testing.assert_array_equal(_twice_image, _image)
np.testing.assert_allclose(_twice_boxes, _boxes, err_msg='Два отражения должны вернуть исходные рамки.')
_empty_image, _empty_boxes = hflip_sample(_image, np.empty((0, 4)))
assert isinstance(_empty_boxes, np.ndarray) and _empty_boxes.shape == (0, 4), 'Сохраните форму пустого набора рамок.'
np.testing.assert_array_equal(_empty_image, _flipped)
print('Задание 1/3: нормирование, отражение, пустые рамки и неизменность входов — OK.')

In [ ]:
def show_domain(item, prediction=None, conf=0.25, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 6))
    with Image.open(item['path']) as im:
        ax.imshow(im.convert('RGB'))
    colors = ['crimson', 'royalblue', 'darkorange']
    if prediction is None:
        boxes, labels = item['boxes'], item['labels']
        scores = None
    else:
        take = np.asarray(prediction['scores']) >= conf
        boxes = np.asarray(prediction['boxes']).reshape(-1, 4)[take]
        labels = np.asarray(prediction['labels'], dtype=int)[take]
        scores = np.asarray(prediction['scores'])[take]
    for j, (box, label) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = box
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color=colors[label], linewidth=1.5))
        caption = CLASS_NAMES[label] + (f' {scores[j]:.2f}' if scores is not None else '')
        ax.text(x1, y1, caption, color='white', fontsize=7,
                bbox=dict(facecolor=colors[label], alpha=.7, pad=.4))
    ax.set_title(item['name'] + (' · разметка' if prediction is None else ' · предсказания'))
    ax.axis('off')
    return ax

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for item, ax in zip(train_items[:2], axes):
    show_domain(item, ax=ax)
plt.tight_layout(); plt.show(); plt.close(fig)

# Экспорт вызывает вашу функцию из задания 1.
def yolo_row(box, label, width, height):
    return [int(label), *box_to_yolo(box, width, height)]

example = train_items[0]
row = yolo_row(example['boxes'][0], example['labels'][0], example['width'], example['height'])
_, xc, yc, bw, bh = row
restored = np.array([(xc-bw/2)*example['width'], (yc-bh/2)*example['height'],
                     (xc+bw/2)*example['width'], (yc+bh/2)*example['height']])
assert np.allclose(restored, example['boxes'][0])
print('Исходная рамка:', example['boxes'][0], '| YOLO:', np.round(row, 4))

In [ ]:
# Визуальная проверка вашей аугментации на реальном train-кадре.
with Image.open(example['path']) as image:
    original = np.asarray(image.convert('RGB'))
flipped, flipped_boxes = hflip_sample(original, example['boxes'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pixels, boxes, title in zip(axes, [original, flipped], [example['boxes'], flipped_boxes],
                                    ['Исходный кадр', 'Результат hflip_sample']):
    ax.imshow(pixels)
    for x1, y1, x2, y2 in boxes:
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime'))
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show(); plt.close(fig)

### Ваша проверка данных

1. Выберите другой train-кадр в вызове `show_domain`. Найдите по рамке каждого
   присутствующего класса. Совпадают ли координаты, цвет и название?
2. Для одной рамки вручную объясните пять чисел YOLO-строки. Как обнаружить
   перепутанные ширину/высоту изображения обратным преобразованием?
3. Запишите, какой класс, по вашему предположению, окажется труднее и почему.
4. Проверьте полноту разметки: есть ли видимые объекты без рамок? Сохраните
   сомнительные примеры для обсуждения; не исправляйте val/test по предсказаниям
   после выбора модели.

| Проверенный кадр | Рамка / класс | Проверка координат | Ожидаемая трудность |
|---|---|---|---|
| … | … | … | … |

Добавьте свой контрольный пример для преобразования рамок и объясните результат.
Ваша функция экспорта используется для подготовки YOLO-данных; отражение — в
загрузчиках SSDLite и YOLOv5. YOLO11/26 используют отражение своей библиотеки.

## 3. Как получаем сопоставимые результаты

Все четыре модели получают одинаковые train/val и классы. Фиксированы seed, 320 пикселей,
число эпох, optimizer AdamW, начальная learning rate и горизонтальное отражение.
У YOLO отключены mosaic, масштабирование и цветовые аугментации, чтобы сравнение
не превращалось в сравнение большого числа настроек. Warmup отключён, learning
rate постоянна: две пробные эпохи не должны целиком уходить на разогрев.

У SSDLite новая голова имеет три класса + фон, метки внутри библиотеки сдвигаются
на 1. У всех YOLO три класса и индексы 0–2. Для всех моделей используется **последняя
эпоха**, а не разные правила выбора лучшего checkpoint.

Оценщик ниже общий: matching одного класса при IoU ≥ 0.5, AP по оболочке precision,
затем среднее трёх AP. Сохраняем score ≥ 0.01, поэтому столбец называется
`mAP50_above_001`; это учебный all-points AP, не полный COCO mAP50–95.
Recall и TP/FP/FN в таблице вычисляются при confidence 0.25. Просмотр ползунком
не меняет этот протокол.

Сравнение показывает работу **четырёх полных систем**: backbone, головы, назначение
соответствий, resize и функции потерь отличаются. SSDLite растягивает вход до
320×320 и фиксирует статистики BatchNorm для маленьких batch; YOLO сохраняет
пропорции через letterbox и замораживает BN только в замороженных модулях.
Разницу AP нельзя целиком объяснить наличием якорей. Абсолютные loss разных моделей
также нельзя сравнивать как общий показатель качества.

У исходного YOLOv5 используются его настоящие якорная голова и функция потерь,
а цикл обучения упрощён: шаг AdamW на каждом batch, без накопления градиентов.
COCO-якоря фиксированы; AutoAnchor не запускается. Функция подготовки автоматически
скачивает нужный код и веса — вручную клонировать репозиторий не требуется.

YOLO26 обучает две головы, а общий оценщик использует one-to-many с NMS.
В закреплённой версии это задаётся `end2end=False` при predict. У YOLO26
`reg_max=1`: столбец с историческим именем `train/dfl_loss` описывает L1-регрессию,
а не Distribution Focal Loss. CSV отражает компоненты one-to-one головы; их
сумма на графике — диагностическая кривая, не полная взвешенная цель двух голов.

### Реализуйте 2/3: какие параметры учатся

Заполните `configure_finetuning`. Она вызывается при подготовке каждой из четырёх
моделей, а её результат передаётся оптимизатору.

1. Найдите параметры backbone по `backbone_prefixes`: это имена модулей, например
   `('backbone',)` или `('model.0.', 'model.1.')`. Проверяйте границу имени:
   `model.1` не должен совпасть с `model.10`. Завершающая точка необязательна.
2. При `freeze_backbone=True` отключите градиенты backbone. При `False` снова
   включите их. Остальные параметры должны обучаться в обоих режимах.
3. Исключение — `always_frozen`: **полные** имена служебных параметров, которые
   должны оставаться замороженными при любом режиме. Например, фиксированный
   слой преобразования координат не становится обучаемым при разморозке backbone.
4. Верните список ровно тех `torch.nn.Parameter`, у которых `requires_grad=True`,
   в порядке `model.named_parameters()`. При неизвестном префиксе backbone или
   имени `always_frozen` вызовите `ValueError`: опечатка не должна менять опыт молча.

Используйте только аргументы функции и обычный Python/PyTorch: функция должна
работать и в отдельном процессе. Не обращайтесь к глобальным объектам моделей.

**Проверка механики:** на небольшой CPU-модели ниже выполняется настоящий шаг
оптимизатора. Предскажите, какие параметры должны измениться. Почему нулевое
изменение одного параметра реальной модели само по себе ещё не доказывает заморозку?
Статистики BatchNorm и режим `train/eval` настраивает готовый цикл обучения отдельно.

In [ ]:
def configure_finetuning(model, backbone_prefixes, freeze_backbone, always_frozen=()):
    """Настроить requires_grad и вернуть обучаемые параметры для оптимизатора."""
    # TODO: проверьте имена; always_frozen имеет приоритет над остальными правилами.
    # TODO: явно установите флаг каждого параметра и верните список обучаемых.
    raise NotImplementedError('Реализуйте configure_finetuning в задании 2/3.')

In [ ]:
class _FreezeToy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torch.nn.Sequential(torch.nn.Linear(2, 3), torch.nn.Linear(3, 3))
        self.head = torch.nn.Linear(3, 1)
        self.backbone_extra = torch.nn.Linear(2, 1, bias=False)
        self.calibration = torch.nn.Parameter(torch.tensor(1.0))
        with torch.no_grad():
            for parameter in self.parameters():
                parameter.fill_(0.2)

    def forward(self, x):
        return self.head(self.backbone(x)) + self.calibration * self.backbone_extra(x)


_toy = _FreezeToy().cpu()
# Начальные флаги намеренно разные: функция должна установить их явно.
_toy.head.bias.requires_grad_(False)
for _freeze in (True, False):
    _params = configure_finetuning(_toy, ('backbone.',), _freeze, ('calibration',))
    assert isinstance(_params, list), 'Для оптимизатора нужен список параметров.'
    _expected_names = [name for name, _ in _toy.named_parameters()
                       if name != 'calibration' and (not _freeze or not name.startswith('backbone.'))]
    _actual_names = [name for name, parameter in _toy.named_parameters() if parameter.requires_grad]
    assert _actual_names == _expected_names, ('Неправильные флаги requires_grad', _freeze, _actual_names)
    assert [id(p) for p in _params] == [id(p) for _, p in _toy.named_parameters() if p.requires_grad], \
        'Список для оптимизатора должен точно соответствовать обучаемым параметрам.'
    _before = {name: p.detach().clone() for name, p in _toy.named_parameters()}
    _optimizer = torch.optim.SGD(_params, lr=0.001)
    _toy.zero_grad(set_to_none=True)
    _toy(torch.ones(2, 2)).square().mean().backward()
    _optimizer.step()
    for _name, _parameter in _toy.named_parameters():
        _changed = not torch.equal(_parameter.detach(), _before[_name])
        assert _changed == (_name in _expected_names), f'После шага optimizer: {_name}, изменился={_changed}.'

# Имена модулей сравниваются по компонентам: backbone_extra — отдельный модуль.
_toy_plain = _FreezeToy().cpu()
configure_finetuning(_toy_plain, ('backbone',), True, ('calibration',))
assert _toy_plain.backbone_extra.weight.requires_grad, 'Префикс backbone не включает backbone_extra.'

_numbered = torch.nn.Module()
_numbered.model = torch.nn.ModuleList([torch.nn.Linear(1, 1) for _ in range(11)])
configure_finetuning(_numbered, ('model.1.',), True)
assert not _numbered.model[1].weight.requires_grad and _numbered.model[10].weight.requires_grad, \
    'Префикс model.1 не должен замораживать model.10.'

for _prefixes, _fixed in [(('backbon_typo',), ()), (('backbone',), ('head.unknown',))]:
    try:
        configure_finetuning(_toy_plain, _prefixes, True, _fixed)
    except ValueError:
        pass
    else:
        raise AssertionError('Неизвестное имя должно вызвать ValueError; проверьте опечатки.')
print('Задание 2/3: freeze/unfreeze, границы имён и реальный шаг оптимизатора — OK.')

### Реализуйте 3/3: сопоставление детекций и разметки

Функция `match_class` работает с **одним классом одного изображения**.
На входе GT-рамки `N×4`, предсказания `M×4` и их confidence `M`.
На выходе — целочисленный NumPy-массив длины `M`: индекс сопоставленной
GT-рамки либо `-1`, если предсказание не имеет пары.

Обрабатывайте предсказания по убыванию confidence, сохраняя исходный порядок
при равных confidence (`stable` сортировка). Для каждого найдите GT с наибольшим
IoU среди **ещё свободных** рамок. Пара принимается при `IoU >= iou_threshold`.
Одну GT-рамку нельзя использовать дважды. При одинаковом IoU выбирайте GT с меньшим
исходным индексом. Итоговый массив верните в **исходном порядке предсказаний**.
Пустые GT и пустые предсказания — обычные случаи. Входные массивы менять нельзя.

Реализуйте IoU внутри функции на NumPy: площади берём из полуоткрытых координат,
для вырожденного объединения считаем IoU равным нулю.

**Предскажите до проверки:** есть одна GT-рамка и два точных предсказания с разными
confidence — сколько получится TP и FP? Почему нужно сортировать по confidence?
Почему высокое confidence ещё не означает правильную локализацию?

Дополните разбор ошибок двумя небольшими функциями:

- `count_matches(matches, n_gt)` возвращает словарь с целочисленными `TP`, `FP`,
  `FN`. Сопоставленное предсказание — TP, несопоставленное — FP,
  неиспользованная GT — FN.
- `select_error_frames(table, limit=3)` получает DataFrame со столбцами
  `image, TP, FP, FN` и возвращает его копию, содержащую не больше `limit`
  худших кадров: сумма `FP + FN` по убыванию, при равенстве — `image` по возрастанию.
  Сохраните исходные столбцы; входную таблицу не меняйте. Это ранжирование по
  абсолютному числу ошибок: кадр со множеством объектов может оказаться первым даже при
  более высокой доле правильных детекций.

После этого готовая обвязка использует ваши функции для метрик всех четырёх
детекторов и для поиска кадров с пропусками и ложными срабатываниями.

In [ ]:
def match_class(gt_boxes, pred_boxes, pred_scores, iou_threshold=0.5):
    """Вернуть для каждого предсказания индекс уникальной GT-пары либо -1."""
    # TODO: stable сортировка по confidence, IoU, выбор ещё свободной GT.
    # TODO: индексы записывайте в исходном порядке предсказаний.
    raise NotImplementedError('Реализуйте match_class в задании 3/3.')


def count_matches(matches, n_gt):
    """Подсчитать TP/FP/FN по уникальным сопоставлениям одного класса."""
    # TODO: посчитайте успешные пары, лишние предсказания и пропущенные GT.
    raise NotImplementedError('Реализуйте count_matches в задании 3/3.')


def select_error_frames(table, limit=3):
    """Выбрать кадры с наибольшим числом FP + FN; входную таблицу не менять."""
    # TODO: ранжируйте с указанным правилом равенства и верните копию строк.
    raise NotImplementedError('Реализуйте select_error_frames в задании 3/3.')

In [ ]:
_gt = np.array([[0., 0., 10., 10.], [20., 0., 30., 10.]])
_pred = np.array([[0., 0., 10., 10.], [20., 0., 30., 10.], [0., 0., 10., 10.]])
_scores = np.array([.2, .7, .9])
_copies = (_gt.copy(), _pred.copy(), _scores.copy())
_matches = match_class(_gt, _pred, _scores)
assert isinstance(_matches, np.ndarray) and _matches.shape == (3,), 'Верните массив длины M.'
assert np.issubdtype(_matches.dtype, np.integer), 'Индексы GT должны быть целыми.'
np.testing.assert_array_equal(_matches, [-1, 1, 0],
    err_msg='Первым пару получает высокое confidence; ответ сохраняет исходный порядок.')
for _original, _copy in zip((_gt, _pred, _scores), _copies):
    np.testing.assert_array_equal(_original, _copy, err_msg='Matching не должен менять входы.')

# Равные confidence: первое предсказание получает единственную GT.
np.testing.assert_array_equal(match_class(_gt[:1], _pred[[0, 2]], [.8, .8]), [0, -1])
# Совпадение ровно на пороге: половина GT, полностью внутри неё, имеет IoU = 0.5.
np.testing.assert_array_equal(match_class(_gt[:1], [[0., 0., 5., 10.]], [.9], .5), [0])
np.testing.assert_array_equal(match_class(_gt[:1], [[0., 0., 4.9, 10.]], [.9], .5), [-1])

# Лучшая GT уже занята, но вторая свободная тоже проходит порог.
_near_gt = np.array([[0., 0., 10., 10.], [2., 0., 12., 10.]])
np.testing.assert_array_equal(match_class(_near_gt, [[0., 0., 10., 10.]] * 2, [.9, .8]), [0, 1],
    err_msg='Ищите лучший IoU среди свободных GT, а не только среди всех GT.')
# Равный IoU с двумя GT: выбираем меньший исходный индекс.
np.testing.assert_array_equal(match_class(np.repeat(_gt[:1], 2, axis=0), _gt[:1], [.8]), [0])
np.testing.assert_array_equal(match_class(np.empty((0, 4)), _pred, _scores), [-1, -1, -1])
_empty_matches = match_class(_gt, np.empty((0, 4)), np.empty(0))
assert _empty_matches.shape == (0,) and np.issubdtype(_empty_matches.dtype, np.integer)
np.testing.assert_array_equal(match_class([[1., 1., 1., 1.]], [[1., 1., 1., 1.]], [.9]), [-1])

assert count_matches(np.array([-1, 1, 0]), 3) == dict(TP=2, FP=1, FN=1)
assert count_matches(np.array([0, 1]), 2) == dict(TP=2, FP=0, FN=0)
assert count_matches(np.array([], dtype=int), 2) == dict(TP=0, FP=0, FN=2)
assert count_matches(np.array([], dtype=int), 0) == dict(TP=0, FP=0, FN=0)
assert count_matches(np.array([-1, -1]), 0) == dict(TP=0, FP=2, FN=0)

import pandas as pd
_error_table = pd.DataFrame([
    dict(image='b.jpg', TP=4, FP=2, FN=1),
    dict(image='c.jpg', TP=2, FP=1, FN=3),
    dict(image='a.jpg', TP=7, FP=0, FN=3),
    dict(image='d.jpg', TP=1, FP=0, FN=0),
])
_original_table = _error_table.copy(deep=True)
_worst = select_error_frames(_error_table, limit=3)
assert isinstance(_worst, pd.DataFrame) and list(_worst.columns) == list(_error_table.columns)
assert list(_worst['image']) == ['c.jpg', 'a.jpg', 'b.jpg'], 'Проверьте FP + FN и разрешение равенств по image.'
pd.testing.assert_frame_equal(_error_table, _original_table)
_worst.loc[_worst.index[0], 'TP'] = -99
pd.testing.assert_frame_equal(_error_table, _original_table, obj='Исходная таблица после изменения результата')
assert len(select_error_frames(_error_table, limit=0)) == 0
assert len(select_error_frames(_error_table.iloc[:0], limit=3)) == 0
assert len(select_error_frames(_error_table, limit=20)) == len(_error_table)
print('Задание 3/3: matching, TP/FP/FN и поиск кадров с ошибками — OK.')

Добавьте по одному собственному контрольному примеру для заморозки и matching.
Объясните ожидаемый результат до выполнения кода. Ваш matching используется
общим оценщиком всех четырёх моделей; ваш выбор параметров — во всех тренерах.
Следующие служебные функции и циклы обучения дописывать не требуется.

In [ ]:
def iou_one_to_many(box, boxes):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    wh = np.maximum(0, np.minimum(box[2:], boxes[:, 2:]) - np.maximum(box[:2], boxes[:, :2]))
    intersection = wh.prod(axis=1)
    union = np.maximum(0, box[2:] - box[:2]).prod() + np.maximum(0, boxes[:, 2:] - boxes[:, :2]).prod(axis=1) - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

def evaluate_domain(subset, predictions, conf=0.25):
    # Один и тот же оценщик для всех моделей. AP: all-points precision envelope.
    class_rows = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        gt = {x['name']: x['boxes'][x['labels'] == class_id] for x in subset}
        n_gt = sum(len(boxes) for boxes in gt.values())
        ranked = []
        for item in subset:
            pred = predictions[item['name']]
            take = (np.asarray(pred['labels']) == class_id) & (np.asarray(pred['scores']) >= LOW_CONF)
            boxes = np.asarray(pred['boxes']).reshape(-1, 4)[take]
            scores = np.asarray(pred['scores'])[take]
            matches = np.asarray(match_class(gt[item['name']], boxes, scores, MATCH_IOU))
            assert matches.shape == scores.shape and np.issubdtype(matches.dtype, np.integer)
            assert np.all((matches >= -1) & (matches < len(gt[item['name']])))
            assert len(set(matches[matches >= 0])) == int(np.sum(matches >= 0)), 'GT использован дважды'
            ranked.extend((float(score), item['name'], int(index >= 0)) for score, index in zip(scores, matches))
        ranked.sort(key=lambda x: -x[0])
        flags = [flag for _, _, flag in ranked]
        tp_prefix = np.cumsum(flags)
        precision = tp_prefix / np.arange(1, len(flags)+1)
        recall = tp_prefix / n_gt if n_gt else np.zeros(len(flags))
        r = np.r_[0., recall, 1.]
        p = np.r_[0., precision, 0.]
        p = np.maximum.accumulate(p[::-1])[::-1]
        ap = float(np.sum(np.diff(r) * p[1:])) if n_gt else float('nan')
        keep = np.array([score >= conf for score, _, _ in ranked], dtype=bool)
        tp = int(np.asarray(flags, dtype=int)[keep].sum())
        fp, fn = int(keep.sum())-tp, n_gt-tp
        class_rows.append(dict(class_name=class_name, n_gt=n_gt, AP50_above_001=ap,
                               precision=tp/(tp+fp) if tp+fp else 0., recall=tp/n_gt if n_gt else 0., TP=tp, FP=fp, FN=fn))
    table = pd.DataFrame(class_rows)
    summary = dict(mAP50_above_001=float(table.AP50_above_001.mean()),
                   macro_recall=float(table.recall.mean()), **{c: int(table[c].sum()) for c in ['TP', 'FP', 'FN']})
    return summary, table

In [ ]:
import hashlib
import json
import random
import time
from functools import partial
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import (
    SSDLite320_MobileNet_V3_Large_Weights,
    ssdlite320_mobilenet_v3_large,
)
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchvision.transforms.functional import pil_to_tensor


def prefetch_ssd_weights(cache_dir=None):
    """Подготовка до запуска таймера обучения; повторно файл не скачивается."""
    global SSDLITE_WEIGHTS_PATH
    url = SSDLite320_MobileNet_V3_Large_Weights.COCO_V1.url
    folder = Path(cache_dir or (Path(torch.hub.get_dir()) / 'checkpoints'))
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / url.rsplit('/', 1)[-1]
    expected_prefix = path.stem.rsplit('-', 1)[-1]
    if not path.exists():
        temporary = path.with_suffix('.download')
        torch.hub.download_url_to_file(url, str(temporary), hash_prefix=expected_prefix)
        temporary.replace(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != 'a79551df90c79834bcd3bb3845ef9d966b5449a3a9b2833ae8404778ca5d65d2':
        raise ValueError(f'Контрольная сумма весов не совпадает: {path}')
    SSDLITE_WEIGHTS_PATH = path
    print('Веса SSDLite готовы:', path.name)
    return path


def _ssd_architecture(num_classes):
    # num_classes — число объектов домена; индекс 0 внутри SSD — фон.
    return ssdlite320_mobilenet_v3_large(
        weights=None, weights_backbone=None, num_classes=num_classes + 1,
        score_thresh=0.01, nms_thresh=0.5, detections_per_img=100,
    )


def build_ssd(num_classes, device):
    """COCO backbone и регрессия рамок + новая классификация K + фон."""
    if not isinstance(num_classes, int) or num_classes < 1:
        raise ValueError('num_classes должен быть положительным целым числом')
    path = globals().get('SSDLITE_WEIGHTS_PATH')
    if path is None or not Path(path).is_file():
        raise RuntimeError('Сначала выполните prefetch_ssd_weights(...).')
    model = _ssd_architecture(90)  # COCO checkpoint содержит 91 выход с фоном.
    model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
    channels = [block[0][0].in_channels
                for block in model.head.classification_head.module_list]
    anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = SSDLiteClassificationHead(
        channels, anchors, num_classes + 1,
        partial(nn.BatchNorm2d, eps=0.001, momentum=0.03),
    )
    model.to(device)
    print(f'SSDLite320: {sum(p.numel() for p in model.parameters()):,} параметров; '
          f'{num_classes} класса + фон; вход модели 320 × 320.')
    return model


class _SSDDataset(Dataset):
    def __init__(self, items, num_classes, augment_hflip=False):
        self.items = items
        self.num_classes = num_classes
        self.augment_hflip = augment_hflip

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        with Image.open(item['path']) as image:
            image = np.array(image.convert('RGB'))
        boxes = np.asarray(item['boxes'], dtype=np.float32).reshape(-1, 4).copy()
        raw_labels = np.asarray(item['labels'])
        if (raw_labels.ndim != 1 or not np.isfinite(raw_labels).all()
                or not np.equal(raw_labels, raw_labels.astype(np.int64)).all()):
            raise ValueError('labels должны быть одномерным массивом целых индексов')
        labels = torch.as_tensor(raw_labels, dtype=torch.int64).clone()
        if len(boxes) != len(labels) or not np.isfinite(boxes).all():
            raise ValueError('Проверьте количество и координаты рамок')
        height, width = image.shape[:2]
        if len(boxes) and (np.any(boxes[:, 2:] <= boxes[:, :2])
                          or np.any(boxes < 0)
                          or np.any(boxes[:, [0, 2]] > width)
                          or np.any(boxes[:, [1, 3]] > height)):
            raise ValueError('Рамки должны иметь положительную площадь и лежать внутри изображения')
        if len(labels) and (labels.min() < 0 or labels.max() >= self.num_classes):
            raise ValueError('Индекс класса за пределами class_names')
        if self.augment_hflip and random.random() < 0.5:
            image, boxes = hflip_sample(image, boxes)
        tensor = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))).float().div_(255.0)
        boxes = torch.as_tensor(np.ascontiguousarray(boxes), dtype=torch.float32).reshape(-1, 4)
        # Рамки остаются в исходных пикселях. Встроенный transform модели
        # согласованно изменяет размер изображений и рамок, затем нормализует RGB.
        return tensor, {'boxes': boxes, 'labels': labels + 1}


def _ssd_collate(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


def _ssd_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def _ssd_sync(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)


def _ssd_parameter_audit_start(model, parameters, freeze_backbone):
    """Проверяем настройку до обучения; храним только хеш каждого параметра."""
    named = list(model.named_parameters())
    expected = {id(p) for name, p in named
                if not (freeze_backbone and name.startswith('backbone.'))}
    if not isinstance(parameters, list):
        raise AssertionError('configure_finetuning должна вернуть список параметров')
    actual = [id(p) for p in parameters]
    if len(actual) != len(set(actual)) or set(actual) != expected:
        raise AssertionError('configure_finetuning: optimizer должен получить ровно обучаемые параметры')
    if {id(p) for _, p in named if p.requires_grad} != expected:
        raise AssertionError('configure_finetuning: проверьте requires_grad для backbone и головы')
    return {name: hashlib.sha256(p.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes()).hexdigest()
            for name, p in named}


def _ssd_parameter_audit_finish(model, before, run_dir):
    report = dict(optimizer_step=1, frozen_parameters=0, trainable_parameters=0,
                  frozen_changed=0, trainable_changed=0,
                  frozen_changed_examples=[], frozen_unchanged_examples=[],
                  trainable_changed_examples=[], trainable_unchanged_examples=[])
    for name, parameter in model.named_parameters():
        digest = hashlib.sha256(parameter.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes()).hexdigest()
        group = 'trainable' if parameter.requires_grad else 'frozen'
        changed = digest != before[name]
        report[group + '_parameters'] += 1
        if changed:
            report[group + '_changed'] += 1
        examples = report[group + ('_changed_examples' if changed else '_unchanged_examples')]
        if len(examples) < 5:
            examples.append(name)
    (Path(run_dir) / 'step_audit.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    if report['frozen_changed'] != 0 or report['trainable_changed'] == 0:
        raise AssertionError('Проверка первого шага не пройдена: см. step_audit.json')
    print('Первый шаг: замороженных параметров изменилось 0; обучаемых —', report['trainable_changed'])
    return report


def fit_ssd(train_items, val_items, class_names, run_dir, config):
    """Свежий запуск из COCO. Возвращает (last.pt, history); val не участвует в loss."""
    cfg = dict(epochs=2, batch_size=2, lr=0.001, seed=42, device='cpu',
               freeze_backbone=True, augment_hflip=True)
    cfg.update(config)
    for key in ('epochs', 'batch_size'):
        if not isinstance(cfg[key], int) or cfg[key] < 1:
            raise ValueError(f'{key} должен быть положительным целым числом')
    if cfg['lr'] <= 0 or not np.isfinite(cfg['lr']):
        raise ValueError('lr должен быть положительным конечным числом')
    if not train_items or not class_names:
        raise ValueError('Нужны обучающие изображения и class_names')
    if len(set(class_names)) != len(class_names):
        raise ValueError('Имена классов должны быть уникальны')
    train_names = {str(Path(item['path']).resolve()) for item in train_items}
    val_names = {str(Path(item['path']).resolve()) for item in val_items}
    if train_names & val_names:
        raise ValueError('train и val пересекаются')
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = run_dir / 'last.pt'
    if checkpoint.exists():
        raise FileExistsError('Этот запуск уже сохранён. Выберите новое имя run_dir.')
    _ssd_seed(cfg['seed'])
    device = torch.device(cfg['device'])
    model = build_ssd(len(class_names), device)
    parameters = configure_finetuning(model, ('backbone.',), cfg['freeze_backbone'])
    audit_before = _ssd_parameter_audit_start(model, parameters, cfg['freeze_backbone'])
    optimizer = torch.optim.AdamW(parameters, lr=cfg['lr'], weight_decay=0.0005)
    trainable = sum(p.numel() for p in parameters)
    print(f'Обучаются {trainable:,} параметров. BN statistics фиксированы; '
          'обучаемые affine-параметры BN получают градиенты.')
    dataset = _SSDDataset(train_items, len(class_names), cfg['augment_hflip'])
    history = []
    total_start = time.perf_counter()
    for epoch in range(1, cfg['epochs'] + 1):
        _ssd_seed(cfg['seed'] + epoch)
        generator = torch.Generator().manual_seed(cfg['seed'] + epoch)
        loader = DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True,
                            num_workers=0, collate_fn=_ssd_collate, generator=generator)
        model.train()
        if cfg['freeze_backbone']:
            model.backbone.eval()
        # Маленький batch, включая последний batch из одной картинки:
        # не обновляем BN mean/variance, affine головы остаётся обучаемым.
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        _ssd_sync(device)
        epoch_start = time.perf_counter()
        totals = {}
        seen = 0
        for images, targets in loader:
            images = [image.to(device) for image in images]
            targets = [{key: value.to(device) for key, value in target.items()}
                       for target in targets]
            optimizer.zero_grad(set_to_none=True)
            losses = model(images, targets)
            loss = sum(losses.values())
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Неконечный loss в эпохе {epoch}: {losses}')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(parameters, max_norm=10.0, error_if_nonfinite=True)
            optimizer.step()
            if audit_before is not None:
                _ssd_parameter_audit_finish(model, audit_before, run_dir)
                audit_before = None
            batch_size = len(images)
            for name, value in {**losses, 'loss': loss}.items():
                totals[name] = totals.get(name, 0.0) + float(value.detach()) * batch_size
            seen += batch_size
        _ssd_sync(device)
        row = dict(epoch=epoch, **{name: value / seen for name, value in totals.items()},
                   elapsed_seconds=time.perf_counter() - epoch_start,
                   lr=optimizer.param_groups[0]['lr'], images=seen)
        history.append(row)
        payload = dict(model={key: value.detach().cpu() for key, value in model.state_dict().items()},
                       optimizer=optimizer.state_dict(), epoch=epoch, class_names=list(class_names),
                       config={**cfg, 'device': str(device)}, history=history,
                       parameters=sum(p.numel() for p in model.parameters()),
                       trainable_parameters=trainable, bn_statistics_frozen=True,
                       checkpoint_selection='last_epoch', torch_version=str(torch.__version__))
        temporary = checkpoint.with_suffix('.tmp')
        torch.save(payload, temporary)
        temporary.replace(checkpoint)
        (run_dir / 'history.json').write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f"Эпоха {epoch}/{cfg['epochs']}: loss={row['loss']:.4f}; "
              f"{row['elapsed_seconds']:.1f} с; сохранён last.pt")
    print(f'Обучение и сохранение: {time.perf_counter() - total_start:.1f} с. '
          'Для общего сравнения используем последнюю эпоху.')
    del model, optimizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return checkpoint, history


def predict_ssd(items, checkpoint, device, conf=0.01, nms_iou=0.5, max_det=100):
    """Готовые предсказания в исходных пикселях; классы снова 0..K−1."""
    if not 0 <= conf <= 1 or not 0 <= nms_iou <= 1 or max_det < 1:
        raise ValueError('Проверьте conf, nms_iou и max_det')
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    model = _ssd_architecture(len(saved['class_names']))
    model.load_state_dict(saved['model'])
    model.score_thresh = conf
    model.nms_thresh = nms_iou
    model.detections_per_img = int(max_det)
    model.to(device).eval()
    predictions = {}
    with torch.inference_mode():
        for item in items:
            name = str(item.get('name', Path(item['path']).name))
            if name in predictions:
                raise ValueError(f'Повтор имени изображения: {name}')
            with Image.open(item['path']) as image:
                tensor = pil_to_tensor(image.convert('RGB')).float().div_(255.0)
            result = model([tensor.to(device)])[0]
            boxes = result['boxes'].cpu().numpy()
            scores = result['scores'].cpu().numpy()
            labels = result['labels'].cpu().numpy()
            keep = np.flatnonzero((labels > 0) & (labels <= len(saved['class_names']))
                                  & (scores >= conf))
            keep = keep[np.argsort(-scores[keep], kind='stable')][:max_det]
            predictions[name] = dict(boxes=boxes[keep].copy(), scores=scores[keep].copy(),
                                     labels=(labels[keep] - 1).astype(np.int64))
    del model
    if torch.device(device).type == 'cuda':
        torch.cuda.empty_cache()
    return predictions

In [ ]:
def prepare_yolo_data():
    folder = ROOT / 'yolo_bccd'
    for split, subset in [('train', train_items), ('val', val_items)]:
        image_dir, label_dir = folder / 'images' / split, folder / 'labels' / split
        image_dir.mkdir(parents=True, exist_ok=True); label_dir.mkdir(parents=True, exist_ok=True)
        for item in subset:
            target = image_dir / item['name']
            if not target.exists():
                shutil.copy2(item['path'], target)
            rows = [yolo_row(box, label, item['width'], item['height']) for box, label in zip(item['boxes'], item['labels'])]
            (label_dir / (Path(item['name']).stem + '.txt')).write_text('\n'.join(' '.join(map(str, row)) for row in rows) + '\n')
        assert {p.name for p in image_dir.glob('*.jpg')} == {x['name'] for x in subset}
        # При исправлении вашей функции размер .txt может не измениться.
        # Библиотечный кэш должен перечитать новые координаты, а не только размеры файлов.
        label_dir.with_suffix('.cache').unlink(missing_ok=True)
    data_yaml = folder / 'data.yaml'
    data_yaml.write_text('path: ' + json.dumps(str(folder.resolve())) + '\ntrain: images/train\nval: images/val\nnames: ' + json.dumps(CLASS_NAMES) + '\n')
    return data_yaml

YOLO_WEIGHT_SHA = {'yolo11n': '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1',
                   'yolo26n': '9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef'}
YOLO_RELEASE = {'yolo11n': 'v8.3.0', 'yolo26n': 'v8.4.0'}

def prepare_yolo_weights(name):
    return download_checked(f'https://github.com/ultralytics/assets/releases/download/{YOLO_RELEASE[name]}/{name}.pt',
                            ROOT / 'weights' / (name + '.pt'), YOLO_WEIGHT_SHA[name])

def fit_yolo(name, run_dir, config):
    from ultralytics.models.yolo.detect import DetectionTrainer
    class StudentTrainer(DetectionTrainer):
        def build_optimizer(self, model, *args, **kwargs):
            prefixes = tuple(f'model.{i}.' for i in range(len(model.yaml['backbone'])))
            always = tuple(n for n, _ in model.named_parameters() if '.dfl.' in n)
            selected = configure_finetuning(model, prefixes, config['freeze_backbone'], always)
            selected_ids = {id(p) for p in selected}
            assert selected_ids == {id(p) for p in model.parameters() if p.requires_grad}
            assert len(selected_ids) == len(selected) and selected, 'Проверьте список параметров optimizer'
            self.freeze_layer_names = list(prefixes) if config['freeze_backbone'] else []
            self.freeze_layer_names += ['.dfl.']
            optimizer = super().build_optimizer(model, *args, **kwargs)
            for group in optimizer.param_groups:
                group['params'] = [p for p in group['params'] if id(p) in selected_ids]
            self._step_before = {n: (p.requires_grad, hashlib.sha256(p.detach().cpu().numpy().tobytes()).hexdigest())
                                 for n, p in model.named_parameters()}
            return optimizer

        def optimizer_step(self):
            super().optimizer_step()
            if self._step_before is not None:
                changed, frozen_changed = [], []
                for n, p in self.model.named_parameters():
                    trainable, before = self._step_before[n]
                    if hashlib.sha256(p.detach().cpu().numpy().tobytes()).hexdigest() != before:
                        (changed if trainable else frozen_changed).append(n)
                report = dict(frozen_changed=len(frozen_changed), trainable_changed=len(changed),
                              frozen_parameters=sum(not flag for flag, _ in self._step_before.values()),
                              trainable_parameters=sum(flag for flag, _ in self._step_before.values()),
                              changed_examples=changed[:5], frozen_changed_examples=frozen_changed[:5])
                (Path(run_dir)/'step_audit.json').write_text(json.dumps(report, indent=2))
                assert not frozen_changed and changed, 'Проверьте заморозку и изменение параметров после шага'
                self._step_before = None
    model = YOLO(str(prepare_yolo_weights(name)))
    if name == 'yolo26n':
        assert model.model.end2end and model.model.model[-1].reg_max == 1
    parameter_info = {}
    def audit(trainer):
        parameter_info['parameters'] = sum(p.numel() for p in trainer.model.parameters())
        parameter_info['trainable_parameters'] = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    model.add_callback('on_train_start', audit)
    start = time.perf_counter()
    model.train(trainer=StudentTrainer, data=str(prepare_yolo_data()), epochs=config['epochs'], imgsz=320,
                batch=config['batch_size'], device=config['device'], workers=0,
                freeze=0, optimizer='AdamW',
                lr0=config['lr'], lrf=1.0, weight_decay=0.0005, nbs=config['batch_size'],
                warmup_epochs=0, cos_lr=False, amp=False, seed=config['seed'], deterministic=True,
                fliplr=0.5 if config['augment_hflip'] else 0., flipud=0.,
                hsv_h=0., hsv_s=0., hsv_v=0., degrees=0., translate=0., scale=0., shear=0., perspective=0.,
                mosaic=0., mixup=0., copy_paste=0., close_mosaic=0, cache=False,
                patience=config['epochs']+1, plots=False, save=True, val=True,
                project=str(Path(run_dir).resolve()), name='fit', exist_ok=False, verbose=False)
    elapsed = time.perf_counter()-start
    directory = Path(model.trainer.save_dir)
    history = pd.read_csv(directory/'results.csv')
    history.columns = history.columns.str.strip()
    # Сумма диагностических компонент. У YOLO26 CSV описывает one-to-one голову,
    # а полная цель обучения использует обе головы и веса их вкладов.
    loss_columns = [c for c in history.columns if c.startswith('train/') and c.endswith('_loss')]
    assert loss_columns, 'В журнале не найдены компоненты loss'
    history['loss'] = history[loss_columns].sum(axis=1)
    history['elapsed_seconds'] = history['time']
    result = dict(checkpoint=directory/'weights/last.pt', seconds=elapsed, **parameter_info)
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result, history.to_dict('records')

def predict_yolo(subset, checkpoint, device=DEVICE):
    model = YOLO(str(checkpoint))
    assert [model.names[i] for i in range(len(CLASS_NAMES))] == CLASS_NAMES
    predictions = {}
    for item in subset:
        result = model.predict(str(item['path']), imgsz=320, conf=LOW_CONF, iou=NMS_IOU,
                               device=device, max_det=MAX_DET, quantize=32, end2end=False, verbose=False)[0]
        predictions[item['name']] = dict(boxes=result.boxes.xyxy.cpu().numpy(),
                                        scores=result.boxes.conf.cpu().numpy(), labels=result.boxes.cls.cpu().numpy().astype(int))
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return predictions

In [ ]:
import subprocess, sys, tempfile

YOLOV5_COMMIT = '50a5c985d77dfe449f1e6ca69c234924078c80ac'
YOLOV5_SOURCE_SHA = 'd46352c6207d24844428edc0499a09c4f53e7c27fa4c886ec77d44775f3a92f1'
YOLOV5_WEIGHT_SHA = '4f180cf23ba0717ada0badd6c685026d73d48f184d00fc159c2641284b2ac0a3'

# Сохраняем оригинальную якорную голову и функцию потерь YOLOv5.
# Отдельный процесс позволяет использовать её рядом с современными YOLO.
_YOLOV5_WORKER = r"""
import os, sys, json, random, time, hashlib
from pathlib import Path
os.environ['YOLOv5_AUTOINSTALL'] = 'false'
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
request = json.loads(Path(sys.argv[1]).read_text())
sys.path.insert(0, request['source'])
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from models.yolo import Model, Detect
from utils.loss import ComputeLoss
from utils.augmentations import letterbox
from utils.general import non_max_suppression, scale_boxes

# Те же функции, которые студент написал в ячейках ноутбука.
exec(request['student_source'], globals())

torch.set_num_threads(min(4, os.cpu_count() or 1))

def seed(value):
    random.seed(value); np.random.seed(value); torch.manual_seed(value)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(value)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # У YOLOv5 objectness targets возможны повторяющиеся индексы CUDA-записи.
    # Одного deterministic cuDNN недостаточно: фиксируем и эти операции.
    torch.use_deterministic_algorithms(True)

def sync(device):
    if device.type == 'cuda': torch.cuda.synchronize(device)

def resized_image(path):
    with Image.open(path) as image:
        image = np.asarray(image.convert('RGB'))
    original_shape = image.shape[:2]
    resized, ratio, pad = letterbox(image, new_shape=(320, 320), auto=False,
                                   scaleFill=False, scaleup=True, stride=32)
    return resized, original_shape, ratio, pad

class BloodDataset(Dataset):
    def __init__(self, items, flip):
        self.items, self.flip = items, flip
    def __len__(self): return len(self.items)
    def __getitem__(self, index):
        item = self.items[index]
        image, _, ratio, pad = resized_image(item['path'])
        boxes = np.asarray(item['boxes'], dtype=np.float32).reshape(-1, 4).copy()
        boxes[:, [0, 2]] = boxes[:, [0, 2]] * ratio[0] + pad[0]
        boxes[:, [1, 3]] = boxes[:, [1, 3]] * ratio[1] + pad[1]
        if self.flip and random.random() < .5:
            image, boxes = hflip_sample(image, boxes)
        targets = np.zeros((len(boxes), 6), dtype=np.float32)
        targets[:, 1] = item['labels']
        height, width = image.shape[:2]
        targets[:, 2:6] = np.asarray([box_to_yolo(box, width, height) for box in boxes],
                                     dtype=np.float32).reshape(-1, 4)
        tensor = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))).float() / 255
        return tensor, torch.from_numpy(targets)

def collate(batch):
    images, targets = zip(*batch)
    for index, target in enumerate(targets): target[:, 0] = index
    return torch.stack(images), torch.cat(targets)

def parameter_audit_start(model, parameters, backbone_prefixes, freeze_backbone):
    named = list(model.named_parameters())
    expected = {id(p) for name, p in named
                if not (freeze_backbone and name.startswith(backbone_prefixes))}
    if not isinstance(parameters, list):
        raise AssertionError('configure_finetuning должна вернуть список параметров')
    actual = [id(p) for p in parameters]
    if len(actual) != len(set(actual)) or set(actual) != expected:
        raise AssertionError('configure_finetuning: optimizer должен получить ровно обучаемые параметры')
    if {id(p) for _, p in named if p.requires_grad} != expected:
        raise AssertionError('configure_finetuning: проверьте requires_grad для backbone и головы')
    return {name: hashlib.sha256(p.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes()).hexdigest()
            for name, p in named}

def parameter_audit_finish(model, before, run_dir):
    report = dict(optimizer_step=1, frozen_parameters=0, trainable_parameters=0,
                  frozen_changed=0, trainable_changed=0,
                  frozen_changed_examples=[], frozen_unchanged_examples=[],
                  trainable_changed_examples=[], trainable_unchanged_examples=[])
    for name, parameter in model.named_parameters():
        digest = hashlib.sha256(parameter.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes()).hexdigest()
        group = 'trainable' if parameter.requires_grad else 'frozen'
        changed = digest != before[name]
        report[group + '_parameters'] += 1
        if changed:
            report[group + '_changed'] += 1
        examples = report[group + ('_changed_examples' if changed else '_unchanged_examples')]
        if len(examples) < 5:
            examples.append(name)
    (Path(run_dir) / 'step_audit.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    if report['frozen_changed'] != 0 or report['trainable_changed'] == 0:
        raise AssertionError('Проверка первого шага не пройдена: см. step_audit.json')
    print('YOLOv5n: первый шаг — замороженных параметров изменилось 0; обучаемых —',
          report['trainable_changed'], flush=True)
    return report

def train():
    config = request['config']
    device = torch.device(config['device'])
    seed(config['seed'])
    start = time.perf_counter()
    pretrained = torch.load(request['weights'], map_location='cpu', weights_only=False)['model'].float()
    architecture = pretrained.yaml
    model = Model(architecture, ch=3, nc=len(request['class_names']))
    source_state, target_state = pretrained.state_dict(), model.state_dict()
    transferred = {key: value for key, value in source_state.items()
                   if key in target_state and value.shape == target_state[key].shape}
    model.load_state_dict(transferred, strict=False)
    del pretrained, source_state, target_state, transferred
    assert isinstance(model.model[-1], Detect) and model.model[-1].na == 3
    backbone_count = len(model.yaml['backbone'])
    assert backbone_count == 10
    model.to(device)
    backbone_prefixes = tuple(f'model.{index}.' for index in range(backbone_count))
    parameters = configure_finetuning(model, backbone_prefixes, config['freeze_backbone'])
    audit_before = parameter_audit_start(model, parameters, backbone_prefixes, config['freeze_backbone'])
    head = model.model[-1]
    # Масштабирование трёх компонент совпадает с официальным train.py.
    model.hyp = dict(box=.05 * 3/head.nl,
                     cls=.5 * len(request['class_names'])/80 * 3/head.nl,
                     obj=1. * (320/640)**2 * 3/head.nl,
                     cls_pw=1., obj_pw=1., fl_gamma=0., anchor_t=4., label_smoothing=0.)
    model.nc = len(request['class_names'])
    model.names = dict(enumerate(request['class_names']))
    criterion = ComputeLoss(model)
    # AdamW: нормы и bias без decay; один шаг на каждый minibatch, warmup отсутствует.
    decay, no_decay = [], []
    selected_ids = {id(parameter) for parameter in parameters}
    for name, parameter in model.named_parameters():
        if id(parameter) in selected_ids:
            (no_decay if parameter.ndim < 2 or name.endswith('.bias') else decay).append(parameter)
    optimizer = torch.optim.AdamW([dict(params=decay, weight_decay=.0005),
                                   dict(params=no_decay, weight_decay=0.)],
                                  lr=config['lr'], betas=(.937, .999))
    parameter_info = dict(parameters=sum(p.numel() for p in model.parameters()),
                          trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad))
    history = []
    run_dir = Path(request['run_dir'])
    checkpoint = run_dir / 'last.pt'
    if checkpoint.exists(): raise FileExistsError('Этот запуск уже сохранён: ' + str(checkpoint))
    anchors_pixels = (head.anchors * head.stride.view(-1, 1, 1)).detach().cpu().tolist()
    for epoch in range(1, config['epochs'] + 1):
        seed(config['seed'] + epoch)
        loader = DataLoader(BloodDataset(request['items'], config['augment_hflip']),
                            batch_size=config['batch_size'], shuffle=True, num_workers=0,
                            collate_fn=collate, generator=torch.Generator().manual_seed(config['seed'] + epoch))
        model.train()
        if config['freeze_backbone']:
            for block in list(model.model)[:backbone_count]: block.eval()
        sync(device); epoch_start = time.perf_counter()
        totals = np.zeros(3); seen = steps = 0
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss, components = criterion(model(images), targets)
            if not torch.isfinite(loss): raise FloatingPointError('Неконечный YOLOv5 loss')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(decay + no_decay, 10., error_if_nonfinite=True)
            optimizer.step()
            if audit_before is not None:
                parameter_audit_finish(model, audit_before, run_dir)
                audit_before = None
            batch = len(images)
            totals += components.detach().cpu().numpy() * batch
            seen += batch; steps += 1
        sync(device)
        values = totals / seen
        row = dict(epoch=epoch, box_loss=float(values[0]), obj_loss=float(values[1]),
                   cls_loss=float(values[2]), loss=float(values.sum()),
                   elapsed_seconds=time.perf_counter()-epoch_start,
                   lr=optimizer.param_groups[0]['lr'], images=seen, optimizer_steps=steps)
        history.append(row)
        saved = dict(model={key: value.detach().cpu() for key, value in model.state_dict().items()},
                     yaml=model.yaml, class_names=request['class_names'], config=config,
                     epoch=epoch, history=history, **parameter_info,
                     source_commit=request['source_commit'], source_sha=request['source_sha'],
                     pretrained_sha=request['pretrained_sha'], anchors_pixels=anchors_pixels,
                     autoanchor=False, frozen_backbone_bn=config['freeze_backbone'],
                     loss_hyperparameters=model.hyp, checkpoint_selection='last_epoch',
                     torch_version=str(torch.__version__))
        temporary = checkpoint.with_suffix('.tmp')
        torch.save(saved, temporary); temporary.replace(checkpoint)
        print(f"YOLOv5n: эпоха {epoch}/{config['epochs']}, loss={row['loss']:.4f}, {row['elapsed_seconds']:.1f} с", flush=True)
    result = dict(checkpoint=str(checkpoint), seconds=time.perf_counter()-start, **parameter_info)
    Path(request['output']).write_text(json.dumps(dict(result=result, history=history)))

def predict():
    device = torch.device(request['device'])
    saved = torch.load(request['checkpoint'], map_location='cpu', weights_only=True)
    assert saved['class_names'] == request['class_names']
    assert saved['source_commit'] == request['source_commit']
    model = Model(saved['yaml'], ch=3, nc=len(saved['class_names']))
    model.load_state_dict(saved['model'])
    model.to(device).eval()
    predictions = {}
    with torch.inference_mode():
        for item in request['items']:
            image, original_shape, ratio, pad = resized_image(item['path'])
            tensor = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))).to(device).float()[None] / 255
            raw = model(tensor)[0]
            detections = non_max_suppression(raw, conf_thres=request['conf'],
                        iou_thres=request['nms_iou'], classes=None, agnostic=False,
                        multi_label=False, max_det=request['max_det'])[0]
            if len(detections):
                detections[:, :4] = scale_boxes(tensor.shape[2:], detections[:, :4],
                                                original_shape, ratio_pad=(ratio, pad))
            values = detections.detach().cpu().numpy()
            predictions[item['name']] = dict(boxes=values[:, :4].tolist(), scores=values[:, 4].tolist(),
                                             labels=values[:, 5].astype(int).tolist())
    Path(request['output']).write_text(json.dumps(predictions))

if request['mode'] == 'train': train()
elif request['mode'] == 'predict': predict()
else: raise ValueError(request['mode'])
"""

def prepare_yolov5():
    folder = ROOT / 'yolov5'
    archive = download_checked(f'https://codeload.github.com/ultralytics/yolov5/zip/{YOLOV5_COMMIT}',
                               folder / 'source.zip', YOLOV5_SOURCE_SHA)
    source = folder / ('yolov5-' + YOLOV5_COMMIT)
    if not (source / '.complete').exists():
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                if not (folder / member.filename).resolve().is_relative_to(folder.resolve()):
                    raise ValueError('Некорректный путь в архиве')
            z.extractall(folder)
        (source / '.complete').touch()
    weights = download_checked('https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5n.pt',
                               folder / 'yolov5n.pt', YOLOV5_WEIGHT_SHA)
    worker = folder / ('worker_' + hashlib.sha256(_YOLOV5_WORKER.encode()).hexdigest()[:12] + '.py')
    worker.write_text(_YOLOV5_WORKER, encoding='utf-8')
    return source.resolve(), weights.resolve(), worker.resolve()

def _yolov5_items(items):
    names = [x['name'] for x in items]
    if len(set(names)) != len(names): raise ValueError('Имена изображений должны быть уникальными')
    return [dict(name=x['name'], path=str(Path(x['path']).resolve()),
                 boxes=np.asarray(x['boxes']).tolist(), labels=np.asarray(x['labels']).tolist()) for x in items]

def _run_yolov5(request, working_dir):
    source, weights, worker = prepare_yolov5()
    request = dict(request, source=str(source), weights=str(weights), class_names=CLASS_NAMES,
                   student_source=student_source(),
                   source_commit=YOLOV5_COMMIT, source_sha=YOLOV5_SOURCE_SHA, pretrained_sha=YOLOV5_WEIGHT_SHA)
    path = Path(working_dir) / 'request.json'
    path.write_text(json.dumps(request), encoding='utf-8')
    log = Path(working_dir) / 'yolov5.log'
    env = dict(os.environ, PYTHONUNBUFFERED='1', MPLBACKEND='Agg', YOLO_AUTOINSTALL='false')
    with log.open('w', encoding='utf-8') as stream:
        process = subprocess.Popen([sys.executable, str(worker), str(path.resolve())],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', env=env)
        try:
            for line in process.stdout:
                stream.write(line); stream.flush()
                if line.startswith('YOLOv5n:'): print(line.strip(), flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate(); process.wait()
            raise
    if returncode:
        raise RuntimeError('Ошибка YOLOv5; полный журнал: ' + str(log) + '\n' + log.read_text()[-3500:])
    return json.loads(Path(request['output']).read_text())

def fit_yolov5(run_dir, config):
    config = dict(config)
    for key in ['epochs', 'batch_size']:
        if not isinstance(config[key], int) or config[key] < 1: raise ValueError(key)
    if not np.isfinite(config['lr']) or config['lr'] <= 0: raise ValueError('lr')
    train_paths = {str(Path(x['path']).resolve()) for x in train_items}
    if train_paths & {str(Path(x['path']).resolve()) for x in val_items}:
        raise ValueError('train и val пересекаются')
    if not train_items: raise ValueError('Обучающий набор пуст')
    run_dir = Path(run_dir).resolve()
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / 'last.pt').exists(): raise FileExistsError('Этот запуск уже сохранён')
    output = run_dir / 'training_result.json'
    result = _run_yolov5(dict(mode='train', run_dir=str(run_dir), config=config,
                             items=_yolov5_items(train_items), output=str(output)), run_dir)
    result['result']['checkpoint'] = Path(result['result']['checkpoint'])
    return result['result'], result['history']

def predict_yolov5(subset, checkpoint, device, conf=.01, nms_iou=.5, max_det=100):
    if not 0 <= conf <= 1 or not 0 <= nms_iou <= 1 or not isinstance(max_det, int) or max_det < 1:
        raise ValueError('Проверьте conf, nms_iou и max_det')
    with tempfile.TemporaryDirectory(prefix='yolov5_predict_') as temporary:
        predictions = _run_yolov5(dict(mode='predict', checkpoint=str(Path(checkpoint).resolve()),
                            device=str(device), conf=conf, nms_iou=nms_iou, max_det=max_det,
                            items=_yolov5_items(subset), output=str(Path(temporary)/'predictions.json')), temporary)
    return {name: dict(boxes=np.asarray(p['boxes'], dtype=float).reshape(-1, 4),
                       scores=np.asarray(p['scores'], dtype=float), labels=np.asarray(p['labels'], dtype=int))
            for name, p in predictions.items()}

In [ ]:
# Передаём ваши функции в отдельный процесс YOLOv5 и учитываем их код в кэше.
def student_source():
    import inspect, textwrap
    return '\n\n'.join(textwrap.dedent(inspect.getsource(function)) for function in
                        [box_to_yolo, hflip_sample, configure_finetuning, match_class,
                         count_matches, select_error_frames])

In [ ]:
def write_json(path, value):
    def convert(value):
        if isinstance(value, Path): return str(value)
        if isinstance(value, np.ndarray): return value.tolist()
        if isinstance(value, np.generic): return value.item()
        raise TypeError(type(value).__name__)
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=convert), encoding='utf-8')

def predict_selected(name, subset, checkpoint, device):
    if name == 'ssdlite320':
        return predict_ssd(subset, checkpoint, device, conf=LOW_CONF, nms_iou=NMS_IOU, max_det=MAX_DET)
    if name == 'yolov5n':
        return predict_yolov5(subset, checkpoint, device, conf=LOW_CONF, nms_iou=NMS_IOU, max_det=MAX_DET)
    return predict_yolo(subset, checkpoint, device)

def protocol(epochs):
    weights = dict(YOLO_WEIGHT_SHA, ssdlite320='a79551df90c79834bcd3bb3845ef9d966b5449a3a9b2833ae8404778ca5d65d2',
                   yolov5n=YOLOV5_WEIGHT_SHA)
    return dict(recipe_version='bccd-v3-student-hooks', models=SELECTED_MODELS, train_config=TRAIN_CONFIG, epochs=epochs, seed=SEED,
                device=DEVICE, class_names=CLASS_NAMES, splits=splits, data_sha=ARCHIVE_SHA,
                low_conf=LOW_CONF, match_iou=MATCH_IOU, nms_iou=NMS_IOU, imgsz=320, max_det=MAX_DET,
                weights={name: weights[name] for name in SELECTED_MODELS},
                model_info={name: MODEL_INFO[name] for name in SELECTED_MODELS},
                yolov5_source_sha=YOLOV5_SOURCE_SHA if 'yolov5n' in SELECTED_MODELS else None,
                yolov5_worker_sha=hashlib.sha256(_YOLOV5_WORKER.encode()).hexdigest() if 'yolov5n' in SELECTED_MODELS else None,
                student_implementation_sha=hashlib.sha256(student_source().encode()).hexdigest(),
                torch=torch.__version__, torchvision=torchvision.__version__,
                ultralytics=ultralytics.__version__, checkpoint='last_epoch')

def protocol_hash(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]

def run_phase(phase, epochs):
    assert phase in {'pilot', 'long'}
    assert 1 <= epochs <= (2 if phase == 'pilot' else 20)
    assert TRAIN_CONFIG['batch_size'] in {1, 2, 4}
    assert 0 < TRAIN_CONFIG['lr'] <= 0.01
    if DEVICE == 'cpu' and not ALLOW_CPU_TRAINING:
        raise RuntimeError('Для обучения включите GPU. На CPU можно изучить данные; отдельный короткий опыт разрешается ALLOW_CPU_TRAINING=True.')
    if phase == 'long' and (not HYPOTHESIS.strip() or not PILOT_NOTE.strip() or not FINAL_CONFIG_NOTE.strip()):
        raise ValueError('Сначала заполните HYPOTHESIS, PILOT_NOTE и FINAL_CONFIG_NOTE после пробного запуска.')
    payload = protocol(epochs)
    phase_dir = ARTIFACTS / (phase + '_' + protocol_hash(payload))
    phase_dir.mkdir(parents=True, exist_ok=True)
    write_json(phase_dir/'protocol.json', payload)
    (phase_dir/'student_functions.py').write_text(student_source(), encoding='utf-8')
    write_json(phase_dir/'notes.json', dict(hypothesis=HYPOTHESIS, pilot=PILOT_NOTE, final=FINAL_CONFIG_NOTE))
    records = []
    for name in SELECTED_MODELS:
        run_dir = phase_dir/name
        run_dir.mkdir(parents=True, exist_ok=True)
        cached = run_dir/'record.json'
        if cached.exists():
            record = json.loads(cached.read_text())
            if not (run_dir/record['checkpoint']).exists():
                raise FileNotFoundError('Не найден сохранённый checkpoint: ' + str(run_dir/record['checkpoint']))
            print(name, ': используется завершённый запуск')
        else:
            print(name, ':', epochs, 'эпох')
            config = dict(TRAIN_CONFIG, epochs=epochs, seed=SEED, device=DEVICE)
            if name == 'ssdlite320':
                prefetch_ssd_weights(ROOT/'weights')
                fit_start = time.perf_counter()
                checkpoint, history = fit_ssd(train_items, val_items, CLASS_NAMES, run_dir, config)
                fit_seconds = time.perf_counter()-fit_start
                saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
                info = {k: saved[k] for k in ['parameters', 'trainable_parameters']}
                result = dict(checkpoint=checkpoint, seconds=fit_seconds, **info)
                del saved
            elif name == 'yolov5n':
                result, history = fit_yolov5(run_dir, config)
            else:
                result, history = fit_yolo(name, run_dir, config)
            checkpoint = Path(result.pop('checkpoint'))
            prediction_start = time.perf_counter()
            predictions = predict_selected(name, val_items, checkpoint, DEVICE)
            prediction_seconds = time.perf_counter()-prediction_start
            metrics, by_class = evaluate_domain(val_items, predictions)
            write_json(run_dir/'predictions_val.json', predictions)
            by_class.to_csv(run_dir/'metrics_by_class.csv', index=False)
            pd.DataFrame(history).to_csv(run_dir/'history.csv', index=False)
            record = dict(model=name, phase=phase, epochs=epochs, protocol_hash=protocol_hash(payload), checkpoint=str(checkpoint.resolve().relative_to(run_dir.resolve())),
                          val_prediction_seconds=prediction_seconds, **result, **metrics)
            write_json(cached, record)
        records.append(dict(record, run_directory=str(run_dir)))
    write_json(phase_dir/'comparison.json', records)
    pd.DataFrame(records).to_csv(phase_dir/'comparison.csv', index=False)
    return records

pilot_records, long_records = [], []

## 4. Пилот на занятии: по две эпохи

Перед запуском проверьте GPU и заполните `HYPOTHESIS`. В конфигурации выше поставьте
`RUN_PILOT=True`, выполните её и следующую ячейку. Переключатель разрешает только
пилот; длинный опыт управляется отдельно.

Дождитесь четырёх моделей. Оставьте `RUN_LONG=False` и выполните раздел 6, чтобы
увидеть таблицу, графики и рамки пилота. После разбора вернитесь к разделу 5.
Проверьте: loss конечный и меняется, нужные параметры
обучаются, файлы весов появились, рамки имеют правильные классы. После двух эпох
модель ещё может быть слабой — это не обязательно ошибка кода.

Если видна проблема, исправьте **одну** настройку, например снимите заморозку
backbone или уменьшите learning rate. Настройка общая для всех четырёх моделей;
повторите пилот один раз и запишите причину. Если явного дефекта нет, объясните,
почему оставляете настройки, либо проверьте одну гипотезу об их изменении.
Улучшение не обязательно.
Не перебирайте десятки комбинаций. Для исправления ошибки разметки сначала
вернитесь к проверке координат, а не повышайте число эпох.

In [ ]:
if RUN_PILOT:
    if not HYPOTHESIS.strip():
        raise ValueError('Запишите HYPOTHESIS до запуска пилота.')
    pilot_records = run_phase('pilot', PILOT_EPOCHS)
else:
    print('Пилот выключен. После проверки данных включите RUN_PILOT.')

## 5. Самостоятельный запуск: по 20 эпох

После просмотра пилота заполните `PILOT_NOTE` и `FINAL_CONFIG_NOTE`. Зафиксируйте
данные и общие настройки для всех четырёх моделей. Выключите `RUN_PILOT`, включите `RUN_LONG`, выполните
ячейку конфигурации, затем ячейку ниже. Полезные функции из предыдущих разделов
должны быть уже выполнены.

Все четыре длинных запуска начинают **с исходных COCO-весов**, а не продолжают пилот.
У каждой модели одинаковый бюджет 20 эпох. Повтор ячейки с тем же протоколом загружает
завершённый результат. Другие параметры создают отдельную папку; не удаляйте
неудачный опыт ради красивого отчёта. Незавершённое обучение автоматически не
возобновляется: сохраните его отдельно и начните новый опыт с другим `STUDENT_ID`.

В Colab заранее задайте `ARTIFACTS` в подключённом Google Drive либо скачайте
папку результатов после завершения. В ней нужны веса, история, настройки и
таблицы: файлы обычной среды Colab могут исчезнуть при её сбросе.

In [ ]:
if RUN_LONG:
    long_records = run_phase('long', LONG_EPOCHS)
else:
    print('Длинный опыт выключен. Включайте его после разбора пилота.')

## 6. Сравнить результат, а не только финальное число

Показаны длинные запуски, если они выполнены, иначе пилоты. Таблица содержит
число параметров, обучаемых параметров, время процедуры обучения и метрики.
Время включает сохранение и действия библиотеки; YOLO также проверяет val во
время обучения. Это время запуска в данном окружении, не чистая скорость forward.
`val_prediction_seconds` включает загрузку модели и предсказания всего val.

По сумме отображаемых компонент loss смотрите динамику **каждой модели отдельно**. По AP классов
проверьте, не прячется ли провал на тромбоцитах за качеством на эритроцитах.
На изображениях ниже сравните все четыре модели при одном confidence: показаны
разметка и четыре предсказания одного кадра. Ползунок меняет порог сразу для всех.

In [ ]:
current_long = [r for r in long_records if r.get('protocol_hash') == protocol_hash(protocol(LONG_EPOCHS))]
current_pilot = [r for r in pilot_records if r.get('protocol_hash') == protocol_hash(protocol(PILOT_EPOCHS))]
records = current_long or current_pilot
if records:
    display(pd.DataFrame(records).drop(columns=['checkpoint', 'run_directory', 'protocol_hash'], errors='ignore'))
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for ax, record in zip(axes.flat, records):
        history = pd.read_csv(Path(record['run_directory'])/'history.csv')
        ax.plot(history['epoch'], history['loss'], marker='.', label=record['model'])
        ax.set(title=record['model'], xlabel='Эпоха', ylabel='Сумма компонент loss')
        ax.grid(alpha=.2)
    for ax in list(axes.flat)[len(records):]: ax.axis('off')
    fig.suptitle('Динамика loss: своя шкала у каждой модели')
    plt.tight_layout(); plt.show(); plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 4))
    for record in records:
        metrics = pd.read_csv(Path(record['run_directory'])/'metrics_by_class.csv')
        ax.plot(metrics['class_name'], metrics['AP50_above_001'], marker='o', label=record['model'])
    ax.set(title='Общий оценщик на val: все четыре модели', xlabel='Класс', ylabel='AP50, score ≥ 0.01', ylim=(0, 1))
    ax.legend(); ax.grid(alpha=.2)
    plt.tight_layout(); plt.show(); plt.close(fig)
else:
    print('После пилота или длинного запуска здесь появятся таблица и графики.')

In [ ]:
def compare_models(index=0, conf=0.25):
    if not records:
        print('Сначала выполните пилот или загрузите завершённый запуск.'); return
    item = val_items[index]
    fig, grid = plt.subplots(3, 2, figsize=(12, 14))
    axes = list(grid.flat)
    show_domain(item, ax=axes[0])
    for ax, record in zip(axes[1:], records):
        predictions = json.loads((Path(record['run_directory'])/'predictions_val.json').read_text())
        show_domain(item, predictions[item['name']], conf=conf, ax=ax)
        ax.set_title(record['model'] + f' · {record["epochs"]} эпох')
    for ax in axes[len(records)+1:]: ax.axis('off')
    plt.tight_layout(); plt.show(); plt.close(fig)

if records:
    compare_models(0, 0.25)
    import ipywidgets as widgets
    controls = dict(index=widgets.Dropdown(options=[(x['name'], i) for i, x in enumerate(val_items)], description='Кадр'),
                    conf=widgets.FloatSlider(value=.25, min=.01, max=.95, step=.01, description='conf', continuous_update=False))
    output = widgets.interactive_output(compare_models, controls)
    display(widgets.VBox([*controls.values(), output]))
else:
    print('Сравнение рамок появится после обучения; ползунок не запускает модель.')

In [ ]:
if records:
    # Отчёты реального первого optimizer.step: counts здесь — число тензоров.
    audits = []
    error_rows = []
    for record in records:
        folder = Path(record['run_directory'])
        audit = json.loads((folder/'step_audit.json').read_text())
        audits.append(dict(model=record['model'], **{k: audit[k] for k in
                      ['frozen_parameters', 'trainable_parameters', 'frozen_changed', 'trainable_changed']}))
        predictions = json.loads((folder/'predictions_val.json').read_text())
        model_errors = []
        for item in val_items:
            pred = predictions[item['name']]
            for label, class_name in enumerate(CLASS_NAMES):
                gt = item['boxes'][item['labels'] == label]
                take = (np.asarray(pred['labels']) == label) & (np.asarray(pred['scores']) >= .25)
                matches = match_class(gt, np.asarray(pred['boxes']).reshape(-1, 4)[take],
                                      np.asarray(pred['scores'])[take], MATCH_IOU)
                counts = count_matches(matches, len(gt))
                model_errors.append(dict(image=item['name'], class_name=class_name, **counts))
        details = pd.DataFrame(model_errors)
        totals = details[['TP', 'FP', 'FN']].sum()
        assert all(int(totals[k]) == record[k] for k in ['TP', 'FP', 'FN']), 'Подсчёт ошибок расходится с общей таблицей'
        by_image = details.groupby('image', as_index=False)[['TP', 'FP', 'FN']].sum()
        worst = select_error_frames(by_image, limit=3)
        details.to_csv(folder/'errors_by_class_image.csv', index=False)
        by_image.to_csv(folder/'errors_by_image.csv', index=False)
        worst.to_csv(folder/'worst_val.csv', index=False)
        error_rows.extend(dict(model=record['model'], **row) for row in by_image.to_dict('records'))
    display(pd.DataFrame(audits))
    print('У замороженных параметров должно быть frozen_changed = 0; у обучаемых trainable_changed > 0.')
    error_table = pd.DataFrame(error_rows)
    print('Ошибки на val при confidence=0.25 по вашему matching и подсчёту:')
    display(error_table.groupby('model', sort=False)[['TP', 'FP', 'FN']].sum())
    # Общие кадры для сравнения: сумма ошибок всех четырёх моделей.
    aggregate = error_table.groupby('image', as_index=False)[['TP', 'FP', 'FN']].sum()
    worst_frames = select_error_frames(aggregate, limit=3)
    display(worst_frames)
    phase_folder = Path(records[0]['run_directory']).parent
    error_table.to_csv(phase_folder/'errors_all_models.csv', index=False)
    worst_frames.to_csv(phase_folder/'selected_error_frames.csv', index=False)
    for image_name in worst_frames['image']:
        index = next(i for i, item in enumerate(val_items) if item['name'] == image_name)
        compare_models(index, conf=.25)
else:
    print('После обучения здесь появятся проверки шага и кадры, выбранные вашим анализом ошибок.')

### Запишите пять коротких выводов

1. Как проверили свои преобразования координат, заморозку и matching? Покажите
   собственные контрольные примеры и отчёты первого шага: какие параметры изменились?
2. Что показал пилот, что вы изменили и почему этого было достаточно для 20 эпох?
3. Какая модель лучше по mAP50, какая — по самому слабому классу? Совпадают ли ответы?
4. Что произошло с временем и числом параметров? Какие различия между моделями
   мешают объяснить результат только якорями или поколением YOLO?
5. Покажите 2–3 val-кадра, найденных вашим анализом ошибок, с результатами всех
   моделей. Объясните конкретные FP/FN и почему меньший loss не решает выбор модели.

Не обязательно получить улучшение на всех классах. Нулевые или неожиданно низкие
метрики — повод проверить классы, координаты, freeze и learning rate; результат
без объяснения не заменяется запуском ещё сотни эпох.

## 7. Итоговый test и сдача

Сначала зафиксируйте выводы и настройки по val. Затем поставьте `RUN_TEST=True`
в конфигурации и выполните ячейку ниже. Test оценивается один раз для всех четырёх моделей;
повторное выполнение читает сохранённые результаты. После test настройки не
подбирают. Сравните итог с val и добавьте одну фразу о том, сохранился ли вывод.

Сдайте ноутбук с реализованными TODO, своими проверками и результатами их выполнения,
папку результатов пилотов и четырёх длинных
запусков, таблицу сравнения с четырьмя строками, пять коротких выводов и 2–3
иллюстрации, на каждой из которых видны результаты всех моделей на одном кадре.
Не нужно включать исходный датасет и предобученные COCO-веса. Сохраните свои
обученные веса отдельно или дайте доступ к ним; не загружайте большие веса в Git.

In [ ]:
# После 20 эпох откройте test один раз для всех четырёх моделей.
if RUN_TEST:
    if not long_records or {r['model'] for r in long_records} != set(SELECTED_MODELS):
        raise ValueError('Сначала завершите длинный запуск всех четырёх моделей.')
    frozen = protocol(LONG_EPOCHS)
    if any(r.get('protocol_hash') != protocol_hash(frozen) for r in long_records):
        raise ValueError('Настройки изменены после обучения: long_records относятся к другому протоколу.')
    frozen['checkpoints'] = {r['model']: sha256_file(Path(r['run_directory'])/r['checkpoint']) for r in long_records}
    lock = ARTIFACTS/'test_protocol.json'
    if lock.exists() and json.loads(lock.read_text()) != frozen:
        raise ValueError('Test уже открыт для другого протокола. Не подбирайте настройки по test.')
    write_json(lock, frozen)
    test_rows = []
    for record in long_records:
        path = ARTIFACTS/('test_' + record['model'] + '.json')
        if path.exists():
            result = json.loads(path.read_text())
        else:
            checkpoint = Path(record['run_directory'])/record['checkpoint']
            predictions = predict_selected(record['model'], test_items, checkpoint, DEVICE)
            summary, table = evaluate_domain(test_items, predictions)
            result = dict(model=record['model'], **summary)
            write_json(path, result)
            table.to_csv(ARTIFACTS/('test_' + record['model'] + '_classes.csv'), index=False)
        test_rows.append(result)
    display(pd.DataFrame(test_rows))
else:
    print('Test закрыт. Его запускают после выбора настроек по val.')